# exp001 - LLaMA 3.2 Stage 1 Baseline v2.2

**实验目标**：测量 LLaMA 3.2 系列模型（1B / 3B）在 Google Colab GPU 上的推理性能与质量基线。

**指标三分类框架**（对应论文三大研究目标）：
- ① **Memory Footprint** (资源开销) → 可部署性 — 模型权重、峰值显存、KV cache
- ② **Inference Efficiency** (推理效率) → 加速 — TTFT / TPOT / tokens/s
- ③ **Model Quality** (模型质量) → 质量保持 — lm-evaluation-harness（MMLU-Pro / GSM8K / HellaSwag / WinoGrande / TruthfulQA）

**结果目录**：`results/exp001/{hw_slug}/{model_slug}/{memory|efficiency|quality}/`

**模态**：text / vision（exp001 仅 text，vision 留 exp002）

## Section 0: Environment & Dependencies

In [ ]:
# Install required dependencies
# transformers: model loading and inference
# accelerate: device_map='auto' for GPU placement
# huggingface_hub: model download and auth
# ipywidgets: interactive configuration UI
# psutil: system memory monitoring
# pillow: image processing
# datasets: benchmark datasets
# lm-eval: lm-evaluation-harness（质量评估标准框架）
!pip install -q transformers accelerate huggingface_hub \
             ipywidgets psutil pillow datasets lm-eval

In [ ]:
# ================================================================
# Path constants -- all paths defined here, not hardcoded elsewhere
# ================================================================
import sys
from pathlib import Path

# Google Drive project root (after mounting)
DRIVE_ROOT      = "/content/drive/MyDrive/EdgeLLM-Systems"

# Model weights cache (persisted on Drive to avoid re-downloading)
MODEL_CACHE_DIR = f"{DRIVE_ROOT}/models/llama32_models"

# Dataset directory (profiling_suite and quality_suite)
DATASET_DIR     = f"{DRIVE_ROOT}/dataset"

# Experiment results output directory
RESULTS_DIR     = f"{DRIVE_ROOT}/results/exp001"

# Project module path (edge_llm_systems added to sys.path after Drive mount)
PROJECT_SRC     = f"{DRIVE_ROOT}"
if PROJECT_SRC not in sys.path:
    sys.path.insert(0, PROJECT_SRC)

print(f"DRIVE_ROOT:      {DRIVE_ROOT}")
print(f"MODEL_CACHE_DIR: {MODEL_CACHE_DIR}")
print(f"DATASET_DIR:     {DATASET_DIR}")
print(f"RESULTS_DIR:     {RESULTS_DIR}")

## Section 1: Mount Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive -- triggers auth dialog
drive.mount("/content/drive")

# Verify project root exists, create if needed
drive_root_path = Path(DRIVE_ROOT)
if not drive_root_path.exists():
    drive_root_path.mkdir(parents=True, exist_ok=True)
    print(f"[Drive] Created: {DRIVE_ROOT}")
else:
    print(f"[Drive] Exists: {DRIVE_ROOT}")

# 只确保 results/exp001/ 基础目录存在
# raw_runs/ 和 group_summary/ 在 Section 6 按模型名动态创建，不在这里预建
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

print("[Drive] Mount complete, directory structure ready")

## Section 2: HuggingFace Authentication

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Read HF Token from Colab Secrets (set via key icon in left sidebar)
# LLaMA 3.2 is gated -- accept terms on HuggingFace website first
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)
print("[HuggingFace] Authentication complete")

## Section 3: Experiment Configuration (Interactive UI)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# ══════════════════════════════════════════════════════════════════════════════
# v2.2 指标三分类 UI（Accordion 折叠版）
#   ① Memory Footprint    ② Inference Efficiency    ③ Model Quality
# 视觉规则：
#   - 每类用独立 Accordion，勾选时自动展开 + 启用；未勾时自动折叠 + 灰化
#   - modality 由 model 自动推断（exp001 模型全是 text-only），不暴露 UI
#   - Vision 区接口预留但锁定（exp002 实现）
# ══════════════════════════════════════════════════════════════════════════════

# ── 模型能力表（text-only / vision-capable）──────────────────────────────────
MODEL_CAPABILITY = {
    "LLaMA-3.2-1B":         "text",
    "LLaMA-3.2-3B":         "text",
    # exp002 启用后追加：
    # "LLaMA-3.2-11B-Vision": "vision",
}

# ── 模型选择（始终顶部显示，modality 自动从此推断）────────────────────────
w_model = widgets.Dropdown(
    options=list(MODEL_CAPABILITY.keys()),
    value="LLaMA-3.2-1B",
    description="Model:",
    style={"description_width": "initial"},
)
w_model_info = widgets.HTML()

def _on_model_change(_=None):
    cap = MODEL_CAPABILITY.get(w_model.value, "text")
    label = "Text-only model" if cap == "text" else "Vision-language model"
    w_model_info.value = (
        f"<i style='color:#888;font-size:90%'>"
        f"→ modality = <b>{cap}</b> ({label}，自动推断)"
        f"</i>"
    )

w_model.observe(_on_model_change, names="value")
_on_model_change()

model_box = widgets.VBox([w_model, w_model_info])

# ──────────────────────────────────────────────────────────────────────────────
# ① Memory Footprint
# ──────────────────────────────────────────────────────────────────────────────
w_enable_mem = widgets.Checkbox(value=True, description="Enable Memory Footprint")
w_mem_prompt_lens = widgets.Text(
    value="64, 128, 256, 512, 1024, 2048",
    description="Prompt lens:", style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
w_mem_gen_lens = widgets.Text(
    value="32, 64, 128",
    description="Gen lens:", style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
w_mem_repeat = widgets.Dropdown(
    options=[1, 3, 5], value=3,
    description="Repeat:", style={"description_width": "initial"},
)
mem_params_box = widgets.VBox([w_enable_mem, w_mem_prompt_lens, w_mem_gen_lens, w_mem_repeat])

# ──────────────────────────────────────────────────────────────────────────────
# ② Inference Efficiency
# ──────────────────────────────────────────────────────────────────────────────
w_enable_eff = widgets.Checkbox(value=True, description="Enable Inference Efficiency")
w_eff_prompt_lens = widgets.Text(
    value="64, 128, 256, 512, 1024, 2048",
    description="Prompt lens:", style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
w_eff_gen_lens = widgets.Text(
    value="32, 64, 128",
    description="Gen lens:", style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
w_eff_repeat = widgets.Dropdown(
    options=[1, 3, 5], value=3,
    description="Repeat:", style={"description_width": "initial"},
)
eff_params_box = widgets.VBox([w_enable_eff, w_eff_prompt_lens, w_eff_gen_lens, w_eff_repeat])

# ──────────────────────────────────────────────────────────────────────────────
# ③ Model Quality
# ──────────────────────────────────────────────────────────────────────────────
w_enable_qual = widgets.Checkbox(value=False, description="Enable Model Quality")
from edge_llm_systems.lm_eval_runner import BENCHMARK_CONFIGS
TEXT_BENCHMARKS = [k for k, v in BENCHMARK_CONFIGS.items()
                   if v.get("benchmark_type", "text") == "text"]
w_benchmarks = widgets.SelectMultiple(
    options=TEXT_BENCHMARKS, value=TEXT_BENCHMARKS,
    description="Benchmarks:", style={"description_width": "initial"},
    layout=widgets.Layout(width="420px", height="110px"),
)
w_quality_seed = widgets.IntText(
    value=42, description="Seed:", style={"description_width": "initial"},
    layout=widgets.Layout(width="180px"),
)
w_quality_batch = widgets.Dropdown(
    options=[("auto", "auto"), ("1", 1), ("4", 4), ("8", 8)],
    value="auto", description="Batch size:", style={"description_width": "initial"},
)
qual_params_box = widgets.VBox([w_enable_qual, w_benchmarks, w_quality_seed, w_quality_batch])

# ──────────────────────────────────────────────────────────────────────────────
# 🔒 Vision 接口预留（exp002 实现，当前整体锁定）
# ──────────────────────────────────────────────────────────────────────────────
w_image_counts = widgets.Text(
    value="1, 4", description="Image counts:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"), disabled=True,
)
w_image_resolutions = widgets.Text(
    value="336, 448", description="Image res:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"), disabled=True,
)
vision_params_box = widgets.VBox([
    widgets.HTML("<i style='color:#888'>🔒 Vision 参数在 exp002（lmms-eval 接入）后启用</i>"),
    w_image_counts, w_image_resolutions,
])

# ──────────────────────────────────────────────────────────────────────────────
# Accordion 折叠分组
# ──────────────────────────────────────────────────────────────────────────────
accordion = widgets.Accordion(children=[
    mem_params_box,
    eff_params_box,
    qual_params_box,
    vision_params_box,
])
accordion.set_title(0, "① Memory Footprint      (资源开销)")
accordion.set_title(1, "② Inference Efficiency  (推理效率)")
accordion.set_title(2, "③ Model Quality         (模型质量)")
accordion.set_title(3, "🔒 Vision Scenarios     (exp002 预留)")

# ── 灰化 + 自动展开/折叠 ────────────────────────────────────────────────────
def _toggle_panel(panel_box, enabled, except_widgets=()):
    for w in panel_box.children:
        if w in except_widgets:
            continue
        if hasattr(w, "disabled"):
            w.disabled = not enabled
    panel_box.layout.opacity = "1.0" if enabled else "0.45"

def _on_enable_change(_=None):
    _toggle_panel(mem_params_box,  w_enable_mem.value,  except_widgets=(w_enable_mem,))
    _toggle_panel(eff_params_box,  w_enable_eff.value,  except_widgets=(w_enable_eff,))
    _toggle_panel(qual_params_box, w_enable_qual.value, except_widgets=(w_enable_qual,))

    expanded = []
    if w_enable_mem.value:  expanded.append(0)
    if w_enable_eff.value:  expanded.append(1)
    if w_enable_qual.value: expanded.append(2)
    accordion.selected_index = expanded[0] if expanded else None

for cb in (w_enable_mem, w_enable_eff, w_enable_qual):
    cb.observe(_on_enable_change, names="value")

_on_enable_change()

# ── 确认按钮 ─────────────────────────────────────────────────────────────────
w_confirm = widgets.Button(
    description="✅ Confirm Config",
    button_style="success",
    layout=widgets.Layout(width="200px"),
)
w_output = widgets.Output()

CONFIG = {}

def _parse_ints(s):
    return [int(x.strip()) for x in s.split(",") if x.strip()]

def _on_confirm(b):
    global CONFIG
    with w_output:
        w_output.clear_output()
        modality = MODEL_CAPABILITY.get(w_model.value, "text")
        CONFIG = {
            "model_key":                  w_model.value,
            "modality":                   modality,                   # 自动推断，不来自 UI
            # 三个测试开关
            "enable_memory":              w_enable_mem.value,
            "enable_efficiency":          w_enable_eff.value,
            "enable_quality":             w_enable_qual.value,
            # Memory 区参数
            "memory_prompt_lengths":      _parse_ints(w_mem_prompt_lens.value),
            "memory_gen_lengths":         _parse_ints(w_mem_gen_lens.value),
            "memory_repeat":              w_mem_repeat.value,
            # Efficiency 区参数
            "efficiency_prompt_lengths":  _parse_ints(w_eff_prompt_lens.value),
            "efficiency_gen_lengths":     _parse_ints(w_eff_gen_lens.value),
            "efficiency_repeat":          w_eff_repeat.value,
            # Vision 接口预留
            "image_counts":               _parse_ints(w_image_counts.value),
            "image_resolutions":          _parse_ints(w_image_resolutions.value),
            "vision_gen_lengths":         _parse_ints(w_eff_gen_lens.value),
            # Quality 参数
            "benchmarks":                 list(w_benchmarks.value),
            "quality_seed":               w_quality_seed.value,
            "quality_batch_size":         w_quality_batch.value,
        }

        yn = lambda v: "✅" if v else "⬜"
        print("✅ Configuration confirmed")
        print(f"   Model    : {CONFIG['model_key']}  (modality auto = {CONFIG['modality']})")
        print()
        print(f"   ① Memory Footprint     : {yn(CONFIG['enable_memory'])}")
        if CONFIG['enable_memory']:
            print(f"      prompt={CONFIG['memory_prompt_lengths']}")
            print(f"      gen   ={CONFIG['memory_gen_lengths']}  repeat={CONFIG['memory_repeat']}")
        print(f"   ② Inference Efficiency : {yn(CONFIG['enable_efficiency'])}")
        if CONFIG['enable_efficiency']:
            print(f"      prompt={CONFIG['efficiency_prompt_lengths']}")
            print(f"      gen   ={CONFIG['efficiency_gen_lengths']}  repeat={CONFIG['efficiency_repeat']}")
        print(f"   ③ Model Quality        : {yn(CONFIG['enable_quality'])}")
        if CONFIG['enable_quality']:
            print(f"      benchmarks={CONFIG['benchmarks']}")
            print(f"      seed={CONFIG['quality_seed']}  batch={CONFIG['quality_batch_size']}")

        if CONFIG['enable_memory'] and CONFIG['enable_efficiency']:
            same = (CONFIG['memory_prompt_lengths'] == CONFIG['efficiency_prompt_lengths']
                    and CONFIG['memory_gen_lengths'] == CONFIG['efficiency_gen_lengths']
                    and CONFIG['memory_repeat'] == CONFIG['efficiency_repeat'])
            if same:
                print("\n   ✨ Memory + Efficiency 参数相同 → s7 将启用单次推理优化（s8 自动跳过）")
            else:
                print("\n   ⚠️  Memory + Efficiency 参数不同 → s7/s8 各跑一次")

w_confirm.on_click(_on_confirm)

display(widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:4px'>Experiment Configuration</h3>"),
    widgets.HTML("<i style='color:#666;font-size:90%'>v2.2 三分类框架（Memory / Efficiency / Quality）</i>"),
    widgets.HTML("<hr>"),
    model_box,
    widgets.HTML("<hr>"),
    accordion,
    widgets.HTML("<hr>"),
    w_confirm,
    w_output,
]))

## Section 4: Model Loading

In [ ]:
import torch
from edge_llm_systems.models import (
    check_model_integrity, download_model_if_needed,
    load_text_model, load_vision_model,
    get_model_load_mem_mb, MODEL_REGISTRY,
)
from edge_llm_systems.utils import log

assert CONFIG, "Please run Section 3 and click Confirm first"

model_key = CONFIG["model_key"]
modality  = CONFIG["modality"]
device    = "cuda" if torch.cuda.is_available() else "cpu"

log(f"[Model Load] Target: {model_key}")
log(f"[Model Load] Modality: {modality}")
log(f"[Model Load] Device: {device}")

local_model_path = download_model_if_needed(
    model_key=model_key,
    cache_dir=MODEL_CACHE_DIR,
    hf_token=HF_TOKEN,
)

if modality == "text":
    model, tokenizer = load_text_model(local_model_path, HF_TOKEN)
    processor = None
else:  # vision — exp002
    model, processor = load_vision_model(local_model_path, HF_TOKEN)
    tokenizer = processor.tokenizer

model_load_mem_mb = get_model_load_mem_mb(device)

cfg = model.config
num_layers   = getattr(cfg, "num_hidden_layers", "N/A")
num_kv_heads = getattr(cfg, "num_key_value_heads",
                        getattr(cfg, "num_attention_heads", "N/A"))

log(f"  Path:     {local_model_path}")
log(f"  Device:   {model.device}")
log(f"  Layers:   {num_layers}")
log(f"  KV Heads: {num_kv_heads}")
log(f"  Base Mem: {model_load_mem_mb:.1f} MB")

## Section 5: Warm-up

In [ ]:
from edge_llm_systems.runners import run_warmup_text, run_warmup_vision
from edge_llm_systems.utils import log

log("[Warmup] Starting (results NOT written to CSV)...")

if modality == "text":
    run_warmup_text(model=model, tokenizer=tokenizer, device=device)
else:
    from PIL import Image as PILImage
    warmup_image = PILImage.new("RGB", (224, 224), color=(128, 128, 128))
    run_warmup_vision(model=model, processor=processor, device=device,
                      warmup_image=warmup_image)

log("[Warmup] Complete")

## Section 6: Experiment Initialization

In [ ]:
from pathlib import Path
import datetime
from edge_llm_systems.categories import (
    CATEGORY_MEMORY, CATEGORY_EFFICIENCY, CATEGORY_QUALITY,
)
from edge_llm_systems.utils import (
    log, generate_run_id, model_hash,
    collect_environment_info, collect_model_info, collect_run_config, save_json,
    get_hw_slug,
)
from edge_llm_systems.models import MODEL_REGISTRY

# ── 实验元数据 ────────────────────────────────────────────────────────────────
run_id     = generate_run_id()
ts         = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
model_id   = MODEL_REGISTRY[model_key]
model_slug = model_id.split("/")[-1]
hw_slug    = get_hw_slug()

# ── 结果目录结构（v2.2 三分类）───────────────────────────────────────────────
# results/exp001/{hw_slug}/{model_slug}/
# ├── memory/       Memory Footprint  CSV  (memory_raw_*.csv / memory_summary_*.csv)
# ├── efficiency/   Inference Efficiency CSV
# ├── quality/      Model Quality CSV     (quality_summary_*.csv, manifest.json)
# └── temp/         lm-eval 原始 JSON
model_result_dir = Path(f"{RESULTS_DIR}/{hw_slug}/{model_slug}")
memory_dir       = model_result_dir / CATEGORY_MEMORY
efficiency_dir   = model_result_dir / CATEGORY_EFFICIENCY
quality_dir      = model_result_dir / CATEGORY_QUALITY
temp_dir         = model_result_dir / "temp"

for d in [memory_dir, efficiency_dir, quality_dir, temp_dir]:
    d.mkdir(parents=True, exist_ok=True)

# ── env_info.json（按硬件存一份，已存在则跳过）────────────────────────────────
env_info_path = Path(f"{RESULTS_DIR}/{hw_slug}/env_info.json")
if not env_info_path.exists():
    env_info = collect_environment_info(
        run_id=run_id, model_id=model_id, model_key=model_key,
        raw_csv_path="", summary_csv_path="",
    )
    save_json(env_info_path, env_info)
    log(f"[Init] env_info     → {env_info_path}")
else:
    log(f"[Init] env_info     已存在，跳过")

# ── model_info.json（每个 model_slug 存一份）─────────────────────────────────
model_info_path = model_result_dir / f"{model_slug}_model_info.json"
if not model_info_path.exists():
    model_info = collect_model_info(
        model=model, model_id=model_id, model_key=model_key,
        local_model_path=local_model_path,
    )
    save_json(model_info_path, model_info)
    log(f"[Init] model_info   → {model_info_path.name}")
else:
    log(f"[Init] model_info   已存在，跳过")

# ── 每行 CSV 附加的元数据 ─────────────────────────────────────────────────────
run_meta = {
    "run_id":     run_id,
    "model_id":   model_id,
    "model_hash": model_hash(model_id),
    "modality":   modality,
}

log(f"[Init] run_id      : {run_id}")
log(f"[Init] timestamp   : {ts}")
log(f"[Init] hw_slug     : {hw_slug}")
log(f"[Init] model_slug  : {model_slug}")
log(f"[Init] result_dir  : {model_result_dir}")
log(f"[Init]   ├── memory/     : {memory_dir.name}/")
log(f"[Init]   ├── efficiency/ : {efficiency_dir.name}/")
log(f"[Init]   ├── quality/    : {quality_dir.name}/")
log(f"[Init]   └── temp/       : {temp_dir.name}/")

## Section 7: ① Memory Footprint Test

测量模型权重显存、推理峰值显存、KV cache 大小（实测 + 理论估算）。

**写入路径**：`results/.../memory/memory_raw_*.csv` + `memory_summary_*.csv`

**单次推理优化**：如果 s3 同时勾选了 Memory 和 Efficiency 且参数相同，本 cell 会一次推理写两份 CSV，s8 自动跳过。

In [ ]:
# ── Section 7: Memory Footprint 测试 ────────────────────────────────────────
from edge_llm_systems.runners import _run_combined_text_loop, _run_combined_vision_loop
from edge_llm_systems.memory_profiler import run_memory_suite_text, run_memory_suite_vision
from edge_llm_systems.utils import log

enable_memory     = CONFIG.get("enable_memory",     False)
enable_efficiency = CONFIG.get("enable_efficiency", False)

# 标志位：s7 是否已经把 efficiency CSV 也一起写了（s8 据此跳过）
_s7_wrote_efficiency = False

if not enable_memory:
    print("[s7-Memory] enable_memory=False，跳过")
else:
    # 检查是否触发"单次推理优化"：两者都勾且参数完全相同
    same_params = (
        enable_efficiency
        and CONFIG["memory_prompt_lengths"] == CONFIG["efficiency_prompt_lengths"]
        and CONFIG["memory_gen_lengths"]    == CONFIG["efficiency_gen_lengths"]
        and CONFIG["memory_repeat"]         == CONFIG["efficiency_repeat"]
    )

    if same_params:
        log("[s7-Memory] Memory + Efficiency 参数相同 → 启用单次推理优化")
        log("[s7-Memory] 同时写入 memory/ 和 efficiency/ 两套 CSV")
        if modality == "text":
            _run_combined_text_loop(
                model=model, tokenizer=tokenizer, device=device,
                prompt_lengths    = CONFIG["memory_prompt_lengths"],
                gen_lengths       = CONFIG["memory_gen_lengths"],
                repeat            = CONFIG["memory_repeat"],
                model_load_mem_mb = model_load_mem_mb,
                results_dir       = model_result_dir,
                ts                = ts,
                run_meta          = run_meta,
            )
        else:
            from PIL import Image as PILImage
            images_by_scenario = {
                (ic, res): PILImage.new("RGB", (res, res), (128, 128, 128))
                for ic in CONFIG["image_counts"]
                for res in CONFIG["image_resolutions"]
            }
            _run_combined_vision_loop(
                model=model, processor=processor, device=device,
                images_by_scenario = images_by_scenario,
                gen_lengths        = CONFIG["vision_gen_lengths"],
                repeat             = CONFIG["memory_repeat"],
                model_load_mem_mb  = model_load_mem_mb,
                results_dir        = model_result_dir,
                ts                 = ts,
                run_meta           = run_meta,
            )
        _s7_wrote_efficiency = True
    else:
        log("[s7-Memory] 单独执行 Memory suite")
        if modality == "text":
            run_memory_suite_text(
                model=model, tokenizer=tokenizer, device=device,
                prompt_lengths    = CONFIG["memory_prompt_lengths"],
                gen_lengths       = CONFIG["memory_gen_lengths"],
                repeat            = CONFIG["memory_repeat"],
                model_load_mem_mb = model_load_mem_mb,
                results_dir       = model_result_dir,
                ts                = ts,
                run_meta          = run_meta,
            )
        else:
            from PIL import Image as PILImage
            images_by_scenario = {
                (ic, res): PILImage.new("RGB", (res, res), (128, 128, 128))
                for ic in CONFIG["image_counts"]
                for res in CONFIG["image_resolutions"]
            }
            run_memory_suite_vision(
                model=model, processor=processor, device=device,
                images_by_scenario = images_by_scenario,
                gen_lengths        = CONFIG["vision_gen_lengths"],
                repeat             = CONFIG["memory_repeat"],
                model_load_mem_mb  = model_load_mem_mb,
                results_dir        = model_result_dir,
                ts                 = ts,
                run_meta           = run_meta,
            )

    log("[s7-Memory] ✅ 完成")

## Section 8: ② Inference Efficiency Test

测量推理延迟（TTFT、TPOT、端到端）与吞吐（tokens/s）。

**写入路径**：`results/.../efficiency/efficiency_raw_*.csv` + `efficiency_summary_*.csv`

**单次推理优化**：如果 s7 已经在 combined 路径里把 efficiency CSV 写过了，本 cell 自动跳过。

In [ ]:
# ── Section 8: Inference Efficiency 测试 ────────────────────────────────────
from edge_llm_systems.efficiency_profiler import (
    run_efficiency_suite_text, run_efficiency_suite_vision,
)
from edge_llm_systems.utils import log

enable_efficiency = CONFIG.get("enable_efficiency", False)

# 检查 s7 是否已经在 combined 路径里写了 efficiency CSV
_already_written = globals().get("_s7_wrote_efficiency", False)

if not enable_efficiency:
    print("[s8-Efficiency] enable_efficiency=False，跳过")
elif _already_written:
    log("[s8-Efficiency] s7 已通过单次推理优化写入 efficiency CSV，跳过本 cell")
else:
    log("[s8-Efficiency] 单独执行 Efficiency suite")
    if modality == "text":
        run_efficiency_suite_text(
            model=model, tokenizer=tokenizer, device=device,
            prompt_lengths    = CONFIG["efficiency_prompt_lengths"],
            gen_lengths       = CONFIG["efficiency_gen_lengths"],
            repeat            = CONFIG["efficiency_repeat"],
            model_load_mem_mb = model_load_mem_mb,
            results_dir       = model_result_dir,
            ts                = ts,
            run_meta          = run_meta,
        )
    else:
        from PIL import Image as PILImage
        images_by_scenario = {
            (ic, res): PILImage.new("RGB", (res, res), (128, 128, 128))
            for ic in CONFIG["image_counts"]
            for res in CONFIG["image_resolutions"]
        }
        run_efficiency_suite_vision(
            model=model, processor=processor, device=device,
            images_by_scenario = images_by_scenario,
            gen_lengths        = CONFIG["vision_gen_lengths"],
            repeat             = CONFIG["efficiency_repeat"],
            model_load_mem_mb  = model_load_mem_mb,
            results_dir        = model_result_dir,
            ts                 = ts,
            run_meta           = run_meta,
        )
    log("[s8-Efficiency] ✅ 完成")

## Section 9: ③ Model Quality Evaluation (lm-evaluation-harness)

使用 EleutherAI lm-evaluation-harness 运行标准基准测试，协议与 Open LLM Leaderboard 一致。

**文本基准（5 个）：**
- **MMLU-Pro** (~504 samples)：大学级多选知识题，5-shot CoT
- **GSM8K** (500 samples)：数学应用题，8-shot CoT
- **HellaSwag** (500 samples)：常识句子补全，10-shot
- **WinoGrande** (500 samples)：代词消歧，5-shot
- **TruthfulQA MC1** (817 samples)：事实性多选题，0-shot

**写入路径**：原始 JSON → `temp/`，提取 CSV → `quality/`

In [ ]:
# ── s8-config：lm-eval 导入验证 ──────────────────────────────────────────────
from edge_llm_systems.lm_eval_runner import (
    BENCHMARK_CONFIGS,
    run_quality_suite,
    __version__,
)

print(f"[Quality] lm_eval_runner 版本: {__version__}")
print(f"[Quality] 支持的基准 ({len(BENCHMARK_CONFIGS)}):")
for name, cfg in BENCHMARK_CONFIGS.items():
    print(f"  {name:<20} {cfg['num_fewshot']}-shot  limit={cfg['limit']:>4}  {cfg['description']}")

In [ ]:
# ── s8-run：质量评估执行（lm-evaluation-harness）─────────────────────────────
from edge_llm_systems.lm_eval_runner import run_quality_suite
from edge_llm_systems.utils import log

enable_quality = CONFIG.get("enable_quality", False)

if not enable_quality:
    print("[Quality] 未启用（enable_quality=False），跳过")
elif modality != "text":
    print(f"[Quality] modality={modality} 的质量评估需 lmms-eval（exp002 实现），跳过")
else:
    batch_size = CONFIG.get("quality_batch_size", "auto")
    log(f"[Quality] 开始质量评估（lm-eval）")
    log(f"[Quality] run_id       : {run_id}")
    log(f"[Quality] model        : {model_id}")
    log(f"[Quality] benchmarks   : {CONFIG['benchmarks']}")
    log(f"[Quality] seed         : {CONFIG['quality_seed']}")
    log(f"[Quality] batch_size   : {batch_size}")
    log(f"[Quality] quality_dir  : {quality_dir}")
    log(f"[Quality] temp_dir     : {temp_dir}")

    quality_results = run_quality_suite(
        benchmarks=CONFIG["benchmarks"],
        model=model,
        tokenizer=tokenizer,
        device=device,
        run_id=run_id,
        model_id=model_id,
        results_dir=model_result_dir,
        seed=CONFIG["quality_seed"],
        batch_size=batch_size,
    )

    log("[Quality] ✅ 质量评估完成")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Section 9c：视觉质量评估 — 🔒 exp002 预留
# ══════════════════════════════════════════════════════════════════════════════
# 工具：lmms-eval（HuggingFace 多模态评估框架）
# 基准：VQAv2 / MMBench / MathVista / TextVQA / DocVQA
# ══════════════════════════════════════════════════════════════════════════════
print("[Quality] 视觉质量评估将在 exp002 中通过 lmms-eval 实现。")

## Section 10: Results Summary

按三分类组织汇总输出：Memory / Efficiency / Quality 各占一段。
读取 `memory/`、`efficiency/`、`quality/` 三个子目录中最新的 CSV。

In [ ]:
import pandas as pd
from pathlib import Path
from edge_llm_systems.categories import CATEGORY_MEMORY, CATEGORY_EFFICIENCY, CATEGORY_QUALITY
from edge_llm_systems.utils import log, read_csv_utf8sig, get_hw_slug
from edge_llm_systems.models import MODEL_REGISTRY

log("[Results] ══════════════ Experiment Summary (v2.2 三分类) ══════════════")

if "model_slug" not in dir() or not model_slug:
    model_slug = MODEL_REGISTRY.get(CONFIG.get("model_key", ""), "").split("/")[-1]
    if not model_slug:
        print("[Results] ⚠️  无法推导 model_slug，请先运行 Section 6")
        raise SystemExit

hw_slug          = get_hw_slug()
model_result_dir = Path(f"{RESULTS_DIR}/{hw_slug}/{model_slug}")
memory_dir       = model_result_dir / CATEGORY_MEMORY
efficiency_dir   = model_result_dir / CATEGORY_EFFICIENCY
quality_dir      = model_result_dir / CATEGORY_QUALITY

# ── 目录树概览 ────────────────────────────────────────────────────────────────
print(f"\n📁 results/exp001/{hw_slug}/{model_slug}/")
for subdir in [CATEGORY_MEMORY, CATEGORY_EFFICIENCY, CATEGORY_QUALITY, "temp"]:
    d = model_result_dir / subdir
    if d.exists():
        files = sorted(f for f in d.iterdir() if f.is_file())
        print(f"  📂 {subdir}/")
        for f in files[:6]:
            size_kb = f.stat().st_size / 1024
            print(f"    {'📊' if f.suffix=='.csv' else '📄'} {f.name}  ({size_kb:.1f} KB)")
        if len(files) > 6:
            print(f"    ... 共 {len(files)} 个文件")


def _print_table(title, csv_dir, glob_pattern, columns, hint=""):
    print("\n" + "─" * 64)
    print(title)
    print("─" * 64)
    if not csv_dir.exists():
        print(f"  目录不存在：{csv_dir}  {hint}")
        return
    files = sorted(csv_dir.glob(glob_pattern))
    if not files:
        print(f"  未找到 CSV  {hint}")
        return
    rows = read_csv_utf8sig(str(files[-1]))
    if not rows:
        print(f"  {files[-1].name} 为空")
        return
    df = pd.DataFrame(rows)
    cols = [c for c in columns if c in df.columns]
    print(f"  Source: {files[-1].name}")
    print(df[cols].to_string(index=False))


# ── ① Memory Footprint ───────────────────────────────────────────────────────
_print_table(
    title="① Memory Footprint  (资源开销)",
    csv_dir=memory_dir,
    glob_pattern="memory_summary_*.csv",
    columns=["group_id", "model_load_mem_mb", "peak_mem_mb",
             "kv_pkv_prefill_mb", "kv_pkv_final_mb", "kv_est_mb",
             "kv_payload_ratio", "status"],
    hint="(请先运行 Section 7)",
)

# ── ② Inference Efficiency ───────────────────────────────────────────────────
_print_table(
    title="② Inference Efficiency  (推理效率)",
    csv_dir=efficiency_dir,
    glob_pattern="efficiency_summary_*.csv",
    columns=["group_id", "ttft_ms", "tpot_ms",
             "total_latency_ms", "tokens_s", "finish_reason", "status"],
    hint="(请先运行 Section 8)",
)

# ── ③ Model Quality ──────────────────────────────────────────────────────────
print("\n" + "─" * 64)
print("③ Model Quality  (模型质量, lm-evaluation-harness)")
print("─" * 64)
qual_files = sorted(quality_dir.glob("quality_summary_*.csv")) if quality_dir.exists() else []
if qual_files:
    rows = read_csv_utf8sig(str(qual_files[-1]))
    if rows:
        df_q     = pd.DataFrame(rows)
        mean_acc = df_q["accuracy"].astype(float).mean()
        print(f"  Source   : {qual_files[-1].name}")
        print(f"  Model    : {rows[0].get('model_id', 'N/A')}")
        print(f"  Mean Acc : {mean_acc:.2f}%\n")
        header = f"  {'Benchmark':<20} {'Accuracy':>9}  {'±stderr':>8}  {'Samples':>8}"
        print(header)
        print("  " + "─" * (len(header) - 2))
        for r in rows:
            stderr_str = f"±{r['stderr']}%" if r.get("stderr") else "  N/A  "
            print(f"  {r['benchmark']:<20} {float(r['accuracy']):>8.2f}%"
                  f"  {stderr_str:>8}  {r['num_samples']:>8}")
else:
    print("  未找到 CSV  (请先运行 Section 9)")

log("[Results] ✅ Summary complete")

## Section 11: Auto-Unassign Runtime

测试完成后 60 秒后自动释放 GPU，节省 Colab Pro 算力配额。

In [ ]:
# ── 测试完成后自动释放 GPU 运行时 ──
import time
print("✅ 测试完成，60 秒后自动断开运行时...")
print("   结果已实时写入 Drive CSV，数据不会丢失。")
time.sleep(60)   # 给 Drive 同步留出时间

from google.colab import runtime
runtime.unassign()   # 释放 GPU，断开运行时